# Debug Walkthrough: Script 4 (rerouting + recovery)

This notebook is a step-by-step debug companion for [scripts/4_rerouting_and_recovery_scenario_loop.py](scripts/4_rerouting_and_recovery_scenario_loop.py).

It mirrors the **main workflow** with explicit checkpoints and diagnostics so you can inspect: inputs, scenario mapping, disrupted OD identification, capacity updates, and optional rerouting execution.

---
**Usage tips**
- Run cells top-to-bottom once.
- Start with `RUN_FLOW_MODEL = False` for fast debugging.
- Increase `SAMPLE_OD_MAX` gradually when validating scale behavior.
- Keep `DEBUG_SCENARIO_ROW` fixed while comparing logic changes.

In [ ]:
from pathlib import Path
import importlib.util
import json
import logging
import gc

import numpy as np
import pandas as pd
import geopandas as gpd
import duckdb
from tqdm.auto import tqdm
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

script4_path = repo_root / "scripts" / "4_rerouting_and_recovery_scenario_loop.py"
assert script4_path.exists(), f"Script 4 not found: {script4_path}"

spec = importlib.util.spec_from_file_location("s4_debug_module", script4_path)
s4 = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(s4)

print("Repo root:", repo_root)
print("Script 4 loaded:", script4_path)
print("Configured base_path from script 4:", s4.base_path)

In [ ]:
# -----------------------------
# Debug configuration
# -----------------------------
import logging
from pathlib import Path
import importlib.util
import sys

# Bootstrap script module if setup cell was not run
if "s4" not in globals():
    repo_root = Path.cwd()
    while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
        repo_root = repo_root.parent

    # Force this workspace source to the front so we don't import from another clone
    repo_src = repo_root / "src"
    if str(repo_src) not in sys.path:
        sys.path.insert(0, str(repo_src))

    # Clear previously imported nird modules from other paths
    for mod_name in list(sys.modules):
        if mod_name == "nird" or mod_name.startswith("nird."):
            del sys.modules[mod_name]

    script4_path = repo_root / "scripts" / "4_rerouting_and_recovery_scenario_loop.py"
    spec = importlib.util.spec_from_file_location("s4_debug_module", script4_path)
    s4 = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(s4)

DEPTH_KEY = 30
FLOOD_KEY = 1
NUM_OF_CHUNK = 8
NUM_OF_CPU = 4

# Use a sample first for quick iteration; set to None for full dataset
SAMPLE_OD_MAX = 5000

# Which row of recovery design table to inspect in detail
DEBUG_SCENARIO_ROW = 0

# Keep False while debugging preprocessing steps
RUN_FLOW_MODEL = False

logging.basicConfig(format="%(asctime)s %(levelname)s %(message)s", level=logging.INFO)

print({
    "DEPTH_KEY": DEPTH_KEY,
    "FLOOD_KEY": FLOOD_KEY,
    "NUM_OF_CHUNK": NUM_OF_CHUNK,
    "NUM_OF_CPU": NUM_OF_CPU,
    "SAMPLE_OD_MAX": SAMPLE_OD_MAX,
    "DEBUG_SCENARIO_ROW": DEBUG_SCENARIO_ROW,
    "RUN_FLOW_MODEL": RUN_FLOW_MODEL,
})

ImportError: cannot import name 'get_results_variant' from 'nird.utils' (C:\Users\alimu\Desktop\Github\DAFNI-NIRD-clone\src\nird\utils.py)

In [ ]:
def banner(msg: str):
    print("\n" + "=" * 100)
    print(msg)
    print("=" * 100)


def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


results_variant = s4.get_results_variant()
out_path = (
    s4.base_path.parent
    / "results"
    / "rerouting_analysis"
    / results_variant
    / str(DEPTH_KEY)
    / str(FLOOD_KEY)
)
out_path.mkdir(parents=True, exist_ok=True)

debug_db_path = s4.base_path / "dbs" / f"debug_recovery_{DEPTH_KEY}_{FLOOD_KEY}.duckdb"
debug_db_path.parent.mkdir(parents=True, exist_ok=True)

print("results_variant:", results_variant)
print("out_path:", out_path)
print("debug_db_path:", debug_db_path)

In [ ]:
banner("Step A: Load recovery + model parameter inputs")

# 1) Flow breakpoints
breakpoint_path = first_existing([
    s4.base_path / "parameters" / "flow_breakpoint_dict.json",
    s4.base_path / "inputs" / "parameters" / "flow_breakpoint_dict.json",
])
assert breakpoint_path is not None, "flow_breakpoint_dict.json not found"
with open(breakpoint_path, "r") as f:
    flow_breakpoint_dict = json.load(f)
print("flow_breakpoint_dict path:", breakpoint_path)
print("flow_breakpoint_dict keys (sample):", list(flow_breakpoint_dict.keys())[:8])

# 2) Recovery design
bridge_recovery_dict, road_recovery_dict, scenarios, conditions = s4.load_scenarios(s4.base_path)
recovery_csv_path = first_existing([
    s4.base_path / "tables" / "recovery design_updated.csv",
    s4.base_path / "inputs" / "tables" / "recovery design_updated.csv",
])
recovery_df = pd.read_csv(recovery_csv_path) if recovery_csv_path else pd.DataFrame()

print("recovery CSV path:", recovery_csv_path)
print("#scenario rows:", len(scenarios))
display(recovery_df.head(10))

# 3) Script-3 damage output
damage_by_edge, direct_damage_total = s4.load_event_damage_from_script3(s4.base_path, FLOOD_KEY)
print("damage_by_edge rows:", len(damage_by_edge))
print("direct_damage_total:", direct_damage_total)
display(damage_by_edge.head(10))

In [ ]:
banner("Step B: Load and normalize odpfc + road_links")

odpfc_path = (
    s4.base_path.parent
    / "results"
    / "disruption_analysis"
    / results_variant
    / "od"
    / f"odpfc_{DEPTH_KEY}_{FLOOD_KEY}.pq"
)

if not odpfc_path.exists():
    base_odpfc_path = s4.base_path.parent / "results" / "base_scenario" / results_variant / "odpfc.pq"
    print("disruption odpfc missing, fallback to:", base_odpfc_path)
    assert base_odpfc_path.exists(), f"Base odpfc missing: {base_odpfc_path}"
    odpfc_path.parent.mkdir(parents=True, exist_ok=True)
    pd.read_parquet(base_odpfc_path).to_parquet(odpfc_path)

disrupted_candidates = pd.read_parquet(odpfc_path)
print("odpfc_path:", odpfc_path)
print("disrupted_candidates shape:", disrupted_candidates.shape)
print("columns:", list(disrupted_candidates.columns))

if SAMPLE_OD_MAX is not None:
    disrupted_candidates = disrupted_candidates.head(SAMPLE_OD_MAX).copy()
    print("using sampled disrupted_candidates shape:", disrupted_candidates.shape)

if "path" in disrupted_candidates.columns:
    disrupted_candidates["path"] = disrupted_candidates["path"].apply(s4.to_edge_id_list)
if "flood_links" in disrupted_candidates.columns:
    disrupted_candidates["flood_links"] = disrupted_candidates["flood_links"].apply(s4.to_edge_id_list)

disrupted_candidates["od_id"] = disrupted_candidates.index

road_links_path = (
    s4.base_path.parent
    / "results"
    / "disruption_analysis"
    / results_variant
    / str(DEPTH_KEY)
    / "links"
    / f"road_links_{FLOOD_KEY}.gpq"
)
assert road_links_path.exists(), f"road_links file missing: {road_links_path}"
road_links = gpd.read_parquet(road_links_path)
road_links["e_id"] = road_links["e_id"].astype(str)
print("road_links_path:", road_links_path)
print("road_links shape:", road_links.shape)
display(road_links.head(5))

In [ ]:
banner("Step C: Merge damage levels and derive flood_links when absent")

if len(damage_by_edge) > 0:
    if "damage_level_max" in road_links.columns:
        road_links = road_links.drop(columns=["damage_level_max"])
    road_links = road_links.merge(damage_by_edge, on="e_id", how="left")
    road_links["damage_level_max"] = road_links["damage_level_max"].fillna("no")

    if "road_label_x" in road_links.columns and "road_label_y" in road_links.columns:
        road_links["road_label"] = road_links["road_label_y"].fillna(road_links["road_label_x"])
        road_links = road_links.drop(columns=["road_label_x", "road_label_y"])

if "road_label" not in road_links.columns:
    road_links["road_label"] = "road"
    if "road_bridge" in road_links.columns:
        mask_bridge = road_links["road_bridge"].astype(str).str.lower() == "yes"
        road_links.loc[mask_bridge, "road_label"] = "bridge"

road_links["breakpoint_flows"] = road_links["combined_label"].map(flow_breakpoint_dict)

if "flood_links" not in disrupted_candidates.columns:
    print("No flood_links in odpfc -> deriving from path x damaged edges")
    flooded_edges = set(road_links.loc[road_links["damage_level_max"] != "no", "e_id"])
    disrupted_candidates["flood_links"] = disrupted_candidates["path"].apply(lambda p: [e for e in p if e in flooded_edges])
    disrupted_candidates = disrupted_candidates[disrupted_candidates["flood_links"].map(len) > 0].reset_index(drop=True)

print("#OD after flood_links filter:", len(disrupted_candidates))
print("damage_level_max distribution:")
print(road_links["damage_level_max"].value_counts(dropna=False))
display(disrupted_candidates[["od_id", "origin_node", "destination_node", "flow", "flood_links"]].head(8))

In [ ]:
banner("Step D: Inspect one recovery scenario row (capacity update logic)")

assert len(scenarios) > 0, "No scenario rows loaded from recovery table"
assert 0 <= DEBUG_SCENARIO_ROW < len(scenarios), "DEBUG_SCENARIO_ROW out of range"

scenario_id = scenarios[DEBUG_SCENARIO_ROW]
event_day = conditions[DEBUG_SCENARIO_ROW]
print("scenario row index:", DEBUG_SCENARIO_ROW)
print("scenario_id:", scenario_id)
print("event_day:", event_day)

road_links_dbg = road_links.copy()
road_links_dbg["acc_capacity"] = road_links_dbg["current_capacity"]

road_links_dbg["acc_capacity"] = road_links_dbg.apply(
    lambda row: (
        s4.bridge_recovery(
            DEBUG_SCENARIO_ROW,
            row["damage_level_max"],
            row["current_capacity"],
            row["acc_capacity"],
            bridge_recovery_dict,
        )
        if row["road_label"] == "bridge"
        else s4.ordinary_road_recovery(
            DEBUG_SCENARIO_ROW,
            row["damage_level_max"],
            row["current_capacity"],
            row["acc_capacity"],
            road_recovery_dict,
        )
    ),
    axis=1,
)

road_links_dbg["capacity_ratio"] = np.where(
    pd.to_numeric(road_links_dbg["current_capacity"], errors="coerce") > 0,
    pd.to_numeric(road_links_dbg["acc_capacity"], errors="coerce") / pd.to_numeric(road_links_dbg["current_capacity"], errors="coerce"),
    np.nan,
)

print("capacity ratio summary:")
print(road_links_dbg["capacity_ratio"].describe())
display(road_links_dbg[["e_id", "road_label", "damage_level_max", "current_capacity", "acc_capacity", "capacity_ratio"]].head(12))

In [ ]:
banner("Step E: Compute disrupted_flow (same bottleneck idea as script 4)")

disrupted_od = disrupted_candidates.copy()
if len(disrupted_od) == 0:
    raise RuntimeError("No disrupted OD pairs found after filtering.")

max_chunk_size = 10_000
total_disrupted = len(disrupted_od)
chunk_size = min(max_chunk_size, total_disrupted)

conn = duckdb.connect(str(debug_db_path))
conn.execute("DROP TABLE IF EXISTS od_results")
first = True

for start in tqdm(range(0, total_disrupted, chunk_size), desc="Debug chunk loop", unit="chunk"):
    chunk = disrupted_od.iloc[start : start + chunk_size].copy()
    chunk = chunk.explode("flood_links")
    if chunk.empty:
        continue

    chunk = chunk.merge(
        road_links_dbg[["e_id", "acc_capacity"]],
        how="left",
        left_on="flood_links",
        right_on="e_id",
    )

    od_df = chunk.groupby("od_id", as_index=False)["acc_capacity"].min()

    if first:
        conn.register("od_df", od_df)
        conn.execute("CREATE TABLE od_results AS SELECT * FROM od_df")
        first = False
    else:
        conn.append("od_results", od_df)

    del chunk, od_df
    gc.collect()

if first:
    conn.close()
    raise RuntimeError("No od_results generated in chunk loop.")

min_capacity = conn.execute("SELECT od_id, MIN(acc_capacity) AS acc_capacity FROM od_results GROUP BY od_id").df()
conn.close()

disrupted_od = disrupted_od.merge(min_capacity, how="left", on="od_id")
disrupted_od["disrupted_flow"] = (
    pd.to_numeric(disrupted_od["flow"], errors="coerce") - pd.to_numeric(disrupted_od["acc_capacity"], errors="coerce")
).clip(lower=0)
disrupted_od = disrupted_od[disrupted_od["disrupted_flow"] > 0].reset_index(drop=True)

print("disrupted_od rows after positive filter:", len(disrupted_od))
print("total disrupted_flow:", float(disrupted_od["disrupted_flow"].sum()))
display(disrupted_od[["od_id", "origin_node", "destination_node", "flow", "acc_capacity", "disrupted_flow"]].head(10))

In [ ]:
banner("Step F: Pre-reroute costs and optional reroute run")

if len(disrupted_od) == 0:
    print("No disrupted OD remains -> no meaningful rerouting outputs for this scenario row.")
else:
    pre_time = (pd.to_numeric(disrupted_od["disrupted_flow"], errors="coerce") * pd.to_numeric(disrupted_od["time_cost_per_flow"], errors="coerce")).sum()
    pre_operate = (pd.to_numeric(disrupted_od["disrupted_flow"], errors="coerce") * pd.to_numeric(disrupted_od["operating_cost_per_flow"], errors="coerce")).sum()
    pre_toll = (pd.to_numeric(disrupted_od["disrupted_flow"], errors="coerce") * pd.to_numeric(disrupted_od["toll_cost_per_flow"], errors="coerce")).sum()
    total_pre = pre_time + pre_operate + pre_toll

    print({
        "pre_time": float(pre_time),
        "pre_operate": float(pre_operate),
        "pre_toll": float(pre_toll),
        "total_pre": float(total_pre),
    })

    if RUN_FLOW_MODEL:
        print("RUN_FLOW_MODEL=True -> executing network_flow_model (can be slow).")

        disrupted_edge_flow = s4.get_flow_on_edges(disrupted_od, "e_id", "path", "disrupted_flow")
        road_links_run = road_links_dbg.merge(disrupted_edge_flow, on="e_id", how="left")
        road_links_run["disrupted_flow"] = road_links_run["disrupted_flow"].fillna(0)

        road_links_run["acc_capacity"] = pd.to_numeric(road_links_run["acc_capacity"], errors="coerce").round(0)
        road_links_run["current_flow"] = pd.to_numeric(road_links_run["current_flow"], errors="coerce").round(0)
        road_links_run["disrupted_flow"] = pd.to_numeric(road_links_run["disrupted_flow"], errors="coerce").round(0)

        road_links_run["acc_capacity"] = road_links_run["acc_capacity"] + road_links_run["disrupted_flow"]
        road_links_run["acc_flow"] = road_links_run["current_flow"] - road_links_run["disrupted_flow"]

        s4.func.update_edge_speed(road_links_run, inplace=True)

        valid_road_links = road_links_run[(road_links_run["acc_capacity"] > 0) & (road_links_run["acc_speed"] > 0)].reset_index(drop=True)
        valid_road_links["from_id"] = valid_road_links["from_id"].astype(str)
        valid_road_links["to_id"] = valid_road_links["to_id"].astype(str)

        network, valid_road_links = s4.func.create_igraph_network(valid_road_links, vehicle_type="car")

        disrupted_od_run = disrupted_od.rename(columns={"disrupted_flow": "Car21"}).copy()
        disrupted_od_run["origin_node"] = disrupted_od_run["origin_node"].astype(str)
        disrupted_od_run["destination_node"] = disrupted_od_run["destination_node"].astype(str)

        iso_path = out_path / f"debug_trip_isolations_s{scenario_id}.pq"
        odpfc_path_iter = out_path / f"debug_odpfc_s{scenario_id}.pq"

        valid_road_links_out, costs = s4.func.network_flow_model(
            valid_road_links,
            network,
            disrupted_od_run[["origin_node", "destination_node", "Car21"]],
            flow_breakpoint_dict,
            NUM_OF_CHUNK,
            NUM_OF_CPU,
            str(debug_db_path),
            iso_out_path=str(iso_path),
            odpfc_out_path=str(odpfc_path_iter),
            vehicle_type="car",
        )

        post_time, post_operate, post_toll, total_post = costs
        print({
            "post_time": float(post_time),
            "post_operate": float(post_operate),
            "post_toll": float(post_toll),
            "total_post": float(total_post),
            "rerouting_cost": float((post_time - pre_time) + (post_operate - pre_operate) + (post_toll - pre_toll)),
        })
    else:
        print("RUN_FLOW_MODEL=False -> stopped before reroute simulation.")

## Suggested debug workflow

1. Validate **input integrity** (Step A/B/C).
2. Pick one `DEBUG_SCENARIO_ROW` and inspect capacity ratios (Step D).
3. Confirm disrupted OD volume exists (Step E).
4. Only then set `RUN_FLOW_MODEL=True` (Step F).
5. Repeat with a different scenario row to compare recovery progression.